In [1]:
# Install Required Libraries
!pip install -q google-genai==1.66.0 chromadb rank-bm25 pymupdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 732.2/732.2 kB 19.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 60.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 86.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 64.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 11.5 MB/s eta 0:00:00
   

In [2]:
import pymupdf

print("PyMuPDF version:", pymupdf.VersionBind)

PyMuPDF version: 1.28.2


In [3]:
# Check Versions
import google.genai
import chromadb
import fitz

print("Google GenAI: OK")
print("ChromaDB:", chromadb.__version__)
print("PyMuPDF:", fitz.VersionBind)

Google GenAI: OK
ChromaDB: 1.5.9
PyMuPDF: 1.28.2


In [4]:
from kaggle_secrets import UserSecretsClient
from google import genai

# Load API key from Kaggle Secrets
user_secrets = UserSecretsClient()

GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY")

# Initialize Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)

print(" Gemini API key loaded successfully!")
print(" Gemini client initialized successfully!")

 Gemini API key loaded successfully!
 Gemini client initialized successfully!


In [5]:
models = list(client.models.list())

print(f"Total models available: {len(models)}\n")

for model in models:
    print(model.name)

Total models available: 52

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gemi

In [6]:
# DOCUMIND AI - GEMINI MODEL CONFIGURATION

from google import genai

# Initialize Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)

# Model Configuration

# Main model for RAG answer generation and multimodal reasoning
GENERATION_MODEL = "gemini-3.6-flash"

# Lightweight model for simple tasks such as query classification
LIGHT_MODEL = "gemini-3.1-flash-lite"

# Model for creating vector embeddings
EMBEDDING_MODEL = "gemini-embedding-2"

print("🚀 DocuMind AI - Model Configuration")
print("-" * 50)
print(f"Generation Model : {GENERATION_MODEL}")
print(f"Light Model      : {LIGHT_MODEL}")
print(f"Embedding Model  : {EMBEDDING_MODEL}")

🚀 DocuMind AI - Model Configuration
--------------------------------------------------
Generation Model : gemini-3.6-flash
Light Model      : gemini-3.1-flash-lite
Embedding Model  : gemini-embedding-2


### Build the PDF Document Processor

In [7]:
# Create a sample upload folder
from pathlib import Path

BASE_DIR = Path("/kaggle/working/documind-ai")

UPLOAD_DIR = BASE_DIR / "data" / "uploads"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Upload directory:", UPLOAD_DIR)
print("Processed directory:", PROCESSED_DIR)

Upload directory: /kaggle/working/documind-ai/data/uploads
Processed directory: /kaggle/working/documind-ai/data/processed


In [8]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith(".pdf"):
            print(os.path.join(root, file))

/kaggle/input/datasets/pranaysoni56/sample/Sample.pdf
/kaggle/input/datasets/pranaysoni56/dataset/documind_sample_report.pdf


In [9]:
PDF_PATH = "/kaggle/input/datasets/pranaysoni56/dataset/documind_sample_report.pdf"

In [10]:
import pymupdf
from pathlib import Path
from PIL import Image
import io


class DocumentProcessor:
    """
    Extracts text, tables, images, and metadata from PDF documents.
    """

    def __init__(self, pdf_path, output_dir):
        self.pdf_path = Path(pdf_path)
        self.output_dir = Path(output_dir)

        self.image_dir = self.output_dir / "images"
        self.image_dir.mkdir(parents=True, exist_ok=True)

    def extract_metadata(self, document):
        """Extract PDF metadata."""

        metadata = document.metadata

        return {
            "title": metadata.get("title", ""),
            "author": metadata.get("author", ""),
            "subject": metadata.get("subject", ""),
            "creator": metadata.get("creator", ""),
            "producer": metadata.get("producer", ""),
            "page_count": len(document)
        }

    def extract_text(self, page):
        """Extract text from a PDF page."""

        text = page.get_text("text")

        return text.strip()

    def extract_tables(self, page):
        """Extract tables from a PDF page."""

        tables = []

        try:
            table_finder = page.find_tables()

            for table_index, table in enumerate(table_finder.tables):

                extracted_table = table.extract()

                tables.append({
                    "table_id": table_index,
                    "data": extracted_table
                })

        except Exception as e:
            print(f"Table extraction warning: {e}")

        return tables

    def extract_images(self, document, page, page_number):
        """Extract images from a PDF page."""

        images = []

        image_list = page.get_images(full=True)

        for image_index, image_info in enumerate(image_list):

            try:
                xref = image_info[0]

                image_data = document.extract_image(xref)

                image_bytes = image_data["image"]
                image_extension = image_data["ext"]

                image_filename = (
                    f"page_{page_number}_image_{image_index}.{image_extension}"
                )

                image_path = self.image_dir / image_filename

                with open(image_path, "wb") as image_file:
                    image_file.write(image_bytes)

                images.append({
                    "image_id": image_index,
                    "image_path": str(image_path),
                    "extension": image_extension
                })

            except Exception as e:
                print(
                    f"Image extraction warning "
                    f"(page {page_number}, image {image_index}): {e}"
                )

        return images

    def process_document(self):
        """Process the complete PDF."""

        document = pymupdf.open(self.pdf_path)

        processed_document = {
            "document_name": self.pdf_path.name,
            "metadata": self.extract_metadata(document),
            "pages": []
        }

        print(f"Processing: {self.pdf_path.name}")
        print(f"Total pages: {len(document)}")

        for page_number, page in enumerate(document, start=1):

            print(f"Processing page {page_number}/{len(document)}")

            page_data = {
                "page_number": page_number,
                "text": self.extract_text(page),
                "tables": self.extract_tables(page),
                "images": self.extract_images(
                    document,
                    page,
                    page_number
                )
            }

            processed_document["pages"].append(page_data)

        document.close()

        return processed_document

In [11]:
processor = DocumentProcessor(
    pdf_path=PDF_PATH,
    output_dir=PROCESSED_DIR
)

document_data = processor.process_document()

Processing: documind_sample_report.pdf
Total pages: 5
Processing page 1/5
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Processing page 2/5
Processing page 3/5
Processing page 4/5
Processing page 5/5


In [12]:
# Inspect the extracted document
print("=" * 60)

print("DOCUMENT:", document_data["document_name"])

print("\nMETADATA:")
for key, value in document_data["metadata"].items():
    print(f"{key}: {value}")

print("\nTOTAL PAGES:", len(document_data["pages"]))

print("=" * 60)

DOCUMENT: documind_sample_report.pdf

METADATA:
title: 
author: 
subject: 
creator: 
producer: 
page_count: 5

TOTAL PAGES: 5


In [13]:
# Check text extraction
for page in document_data["pages"][:3]:

    print("=" * 70)
    print(f"PAGE {page['page_number']}")
    print("=" * 70)

    print(page["text"][:1500])

PAGE 1
DocuMind AI
Quick Sample Business Report for Multimodal RAG Testing
This five-page sample PDF is designed for testing document extraction, chunking, semantic search, tables,
and source citations.
Key Topics
Topic
Purpose
Revenue
Test financial question answering
Customers
Test factual retrieval
Product Innovation
Test keyword and semantic search
Outlook
Test summary and reasoning
PAGE 2
1. Executive Summary
DocuMind Technologies delivered strong performance in Q1 2024. Revenue increased by 25 percent year
over year, reaching $2.45 million. The company added 218 new customers and improved gross profit margin
to 62 percent. Operating expenses decreased by 8.24 percent because of improved efficiency.
Table 1: Key Performance Indicators
Metric
Q1 2024
Q4 2023
Change
Revenue
$2,450,000
$1,960,000
+25%
Gross Profit
$1,519,000
$1,166,000
+30.28%
Gross Profit Margin
62%
59%
+3 pp
Operating Expenses
$890,000
$970,000
-8.24%
New Customers
218
185
+17.84%
PAGE 3
2. Financial Performance
Re

In [14]:
# Check extracted tables
for page in document_data["pages"]:

    if page["tables"]:

        print(f"\n Page {page['page_number']}")

        for table in page["tables"]:

            print("\nTable ID:", table["table_id"])

            for row in table["data"][:5]:
                print(row)


 Page 1

Table ID: 0
['Topic', 'Purpose']
['Revenue', 'Test financial question answering']
['Customers', 'Test factual retrieval']
['Product Innovation', 'Test keyword and semantic search']
['Outlook', 'Test summary and reasoning']

 Page 2

Table ID: 0
['Metric', 'Q1 2024', 'Q4 2023', 'Change']
['Revenue', '$2,450,000', '$1,960,000', '+25%']
['Gross Profit', '$1,519,000', '$1,166,000', '+30.28%']
['Gross Profit Margin', '62%', '59%', '+3 pp']
['Operating Expenses', '$890,000', '$970,000', '-8.24%']

 Page 3

Table ID: 0
['Segment', 'Q1 2024', 'Q4 2023', 'Change']
['Software Solutions', '$1,320,000', '$1,020,000', '+29.41%']
['Consulting Services', '$720,000', '$540,000', '+33.33%']
['Support & Maintenance', '$410,000', '$400,000', '+2.50%']
['Total Revenue', '$2,450,000', '$1,960,000', '+25.00%']

 Page 4

Table ID: 0
['Metric', 'Q1 2024', 'Q4 2023', 'Change']
['Total Customers', '1,258', '1,040', '+20.96%']
['New Customers', '218', '185', '+17.84%']
['Retention Rate', '91.3%', '90.1

In [15]:
# Check extracted images
for page in document_data["pages"]:

    if page["images"]:

        print(f"\n Page {page['page_number']}")

        for image in page["images"]:
            print(image["image_path"])

### Smart Chunking for Multimodal RAG

In [16]:
# Import required libraries
import re
import json
import uuid
from pathlib import Path

In [17]:
# Create the Smart Chunker
class MultimodalChunker:
    """
    Converts processed PDF data into retrieval-ready
    multimodal chunks.
    """

    def __init__(
        self,
        chunk_size=1000,
        chunk_overlap=200
    ):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def clean_text(self, text):
        """Clean extracted PDF text."""

        if not text:
            return ""

        # Remove excessive whitespace
        text = re.sub(r"\s+", " ", text)

        # Remove repeated spaces
        text = re.sub(r" +", " ", text)

        return text.strip()

    def split_text(self, text):
        """
        Split text into chunks while trying to preserve
        paragraph and sentence boundaries.
        """

        text = self.clean_text(text)

        if len(text) <= self.chunk_size:
            return [text] if text else []

        chunks = []

        # Split into sentences
        sentences = re.split(
            r'(?<=[.!?])\s+',
            text
        )

        current_chunk = ""

        for sentence in sentences:

            # If adding the sentence exceeds chunk size
            if (
                len(current_chunk) + len(sentence) + 1
                > self.chunk_size
            ):

                if current_chunk:
                    chunks.append(current_chunk.strip())

                # Add overlap from the previous chunk
                overlap_text = current_chunk[
                    -self.chunk_overlap:
                ]

                current_chunk = (
                    overlap_text + " " + sentence
                )

            else:
                current_chunk += " " + sentence

        if current_chunk.strip():
            chunks.append(current_chunk.strip())

        return chunks

    def table_to_text(self, table_data):
        """
        Convert extracted table data into a text format
        suitable for embeddings.
        """

        if not table_data:
            return ""

        table_text = []

        for row in table_data:

            cleaned_row = [
                str(cell).strip()
                if cell is not None
                else ""
                for cell in row
            ]

            table_text.append(
                " | ".join(cleaned_row)
            )

        return "\n".join(table_text)

    def create_text_chunks(self, page_data, document_name):

        text_chunks = []

        page_number = page_data["page_number"]

        text = page_data.get("text", "")

        chunks = self.split_text(text)

        for index, chunk in enumerate(chunks):

            text_chunks.append({
                "chunk_id": str(uuid.uuid4()),

                "content": chunk,

                "content_type": "text",

                "document_name": document_name,

                "page_number": page_number,

                "chunk_index": index,

                "image_paths": [
                    image["image_path"]
                    for image in page_data.get(
                        "images",
                        []
                    )
                ]
            })

        return text_chunks

    def create_table_chunks(
        self,
        page_data,
        document_name
    ):

        table_chunks = []

        page_number = page_data["page_number"]

        for table in page_data.get(
            "tables",
            []
        ):

            table_text = self.table_to_text(
                table["data"]
            )

            if table_text:

                table_chunks.append({

                    "chunk_id": str(uuid.uuid4()),

                    "content": table_text,

                    "content_type": "table",

                    "document_name": document_name,

                    "page_number": page_number,

                    "table_id": table["table_id"],

                    "image_paths": []
                })

        return table_chunks

    def create_image_chunks(
        self,
        page_data,
        document_name
    ):
        """
        Create metadata chunks for images.

        Actual image understanding will happen later
        using Gemini only when needed.
        """

        image_chunks = []

        page_number = page_data["page_number"]

        for image in page_data.get(
            "images",
            []
        ):

            image_chunks.append({

                "chunk_id": str(uuid.uuid4()),

                "content": (
                    f"Image extracted from page "
                    f"{page_number} of "
                    f"{document_name}"
                ),

                "content_type": "image",

                "document_name": document_name,

                "page_number": page_number,

                "image_id": image["image_id"],

                "image_path": image["image_path"]
            })

        return image_chunks

    def process_document(
        self,
        document_data
    ):
        """
        Convert the entire processed document
        into multimodal chunks.
        """

        all_chunks = []

        document_name = (
            document_data["document_name"]
        )

        for page_data in document_data["pages"]:

            # Text chunks
            text_chunks = self.create_text_chunks(
                page_data,
                document_name
            )

            # Table chunks
            table_chunks = self.create_table_chunks(
                page_data,
                document_name
            )

            # Image chunks
            image_chunks = self.create_image_chunks(
                page_data,
                document_name
            )

            all_chunks.extend(text_chunks)
            all_chunks.extend(table_chunks)
            all_chunks.extend(image_chunks)

        return all_chunks

In [18]:
# Create the chunks
chunker = MultimodalChunker(
    chunk_size=1000,
    chunk_overlap=200
)

multimodal_chunks = chunker.process_document(
    document_data
)

print(
    f" Total chunks created: "
    f"{len(multimodal_chunks)}"
)

 Total chunks created: 9


In [19]:
# Check chunk distribution
from collections import Counter

chunk_types = Counter(
    chunk["content_type"]
    for chunk in multimodal_chunks
)

print(" Chunk Distribution")

for chunk_type, count in chunk_types.items():
    print(f"{chunk_type}: {count}")

 Chunk Distribution
text: 5
table: 4


In [20]:
# Inspect some text chunks
text_chunks = [
    chunk
    for chunk in multimodal_chunks
    if chunk["content_type"] == "text"
]

for chunk in text_chunks[:3]:

    print("=" * 80)

    print(
        f"Chunk ID: "
        f"{chunk['chunk_id']}"
    )

    print(
        f"Page: "
        f"{chunk['page_number']}"
    )

    print(
        f"Type: "
        f"{chunk['content_type']}"
    )

    print("-" * 80)

    print(chunk["content"][:1000])

Chunk ID: 4016dda0-5c1f-4548-b65d-043ebac9ad57
Page: 1
Type: text
--------------------------------------------------------------------------------
DocuMind AI Quick Sample Business Report for Multimodal RAG Testing This five-page sample PDF is designed for testing document extraction, chunking, semantic search, tables, and source citations. Key Topics Topic Purpose Revenue Test financial question answering Customers Test factual retrieval Product Innovation Test keyword and semantic search Outlook Test summary and reasoning
Chunk ID: ef3102b3-4756-42f6-8b75-16b07ed560dd
Page: 2
Type: text
--------------------------------------------------------------------------------
1. Executive Summary DocuMind Technologies delivered strong performance in Q1 2024. Revenue increased by 25 percent year over year, reaching $2.45 million. The company added 218 new customers and improved gross profit margin to 62 percent. Operating expenses decreased by 8.24 percent because of improved efficiency. Table 

In [21]:
# Inspect table chunks
table_chunks = [
    chunk
    for chunk in multimodal_chunks
    if chunk["content_type"] == "table"
]

for chunk in table_chunks[:2]:

    print("=" * 80)

    print(
        f"Page: "
        f"{chunk['page_number']}"
    )

    print(
        f"Table ID: "
        f"{chunk['table_id']}"
    )

    print("-" * 80)

    print(chunk["content"])

Page: 1
Table ID: 0
--------------------------------------------------------------------------------
Topic | Purpose
Revenue | Test financial question answering
Customers | Test factual retrieval
Product Innovation | Test keyword and semantic search
Outlook | Test summary and reasoning
Page: 2
Table ID: 0
--------------------------------------------------------------------------------
Metric | Q1 2024 | Q4 2023 | Change
Revenue | $2,450,000 | $1,960,000 | +25%
Gross Profit | $1,519,000 | $1,166,000 | +30.28%
Gross Profit Margin | 62% | 59% | +3 pp
Operating Expenses | $890,000 | $970,000 | -8.24%
New Customers | 218 | 185 | +17.84%


In [22]:
# Inspect image chunks
image_chunks = [
    chunk
    for chunk in multimodal_chunks
    if chunk["content_type"] == "image"
]

for chunk in image_chunks[:5]:

    print("=" * 80)

    print(
        f"Page: "
        f"{chunk['page_number']}"
    )

    print(
        f"Image Path: "
        f"{chunk['image_path']}"
    )

In [23]:
# Save processed chunks
OUTPUT_FILE = (
    PROCESSED_DIR /
    "multimodal_chunks.json"
)

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        multimodal_chunks,
        file,
        indent=2,
        ensure_ascii=False
    )

print(
    f" Chunks saved successfully to:\n"
    f"{OUTPUT_FILE}"
)

 Chunks saved successfully to:
/kaggle/working/documind-ai/data/processed/multimodal_chunks.json


### Gemini Embeddings + ChromaDB Vector Search

In [24]:
# Create Embedding function

import time
import hashlib
import json
from pathlib import Path


# EMBEDDING CONFIGURATION

EMBEDDING_MODEL = "gemini-embedding-001"

# Local embedding cache
CACHE_FILE = BASE_DIR / "embedding_cache.json"


# LOAD EXISTING CACHE

if CACHE_FILE.exists():

    with open(
        CACHE_FILE,
        "r",
        encoding="utf-8"
    ) as file:

        embedding_cache = json.load(file)

else:

    embedding_cache = {}


print(
    f"Loaded {len(embedding_cache)} cached embeddings."
)


# CREATE UNIQUE TEXT HASH

def get_text_hash(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# SAVE CACHE

def save_embedding_cache():

    with open(
        CACHE_FILE,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            embedding_cache,
            file
        )


# SAFE BATCH EMBEDDING

def get_embeddings_batch(texts):

    """
    Generate embeddings safely.

    Features:
    - Uses cached embeddings
    - Only sends uncached texts to Gemini
    - Makes one API request per batch
    - Stops immediately on 429
    - Does NOT retry automatically
    """

    results = [None] * len(texts)

    texts_to_embed = []
    original_indices = []
    text_hashes = []

    # CHECK CACHE FIRST

    for index, text in enumerate(texts):

        if not text or not text.strip():
            continue

        text = text.strip()

        text_hash = get_text_hash(text)

        # Use cached embedding
        if text_hash in embedding_cache:

            results[index] = embedding_cache[
                text_hash
            ]

        else:

            texts_to_embed.append(text)

            original_indices.append(index)

            text_hashes.append(text_hash)

    # EVERYTHING ALREADY CACHED

    if not texts_to_embed:

        print(
            " All embeddings loaded from cache."
        )

        return results

    # ONE GEMINI API REQUEST

    print(
        f"📡 Sending {len(texts_to_embed)} "
        f"new texts to Gemini..."
    )

    try:

        response = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=texts_to_embed
        )

        new_embeddings = [
            embedding.values
            for embedding in response.embeddings
        ]

    except Exception as e:

        error_message = str(e)

        # STOP IMMEDIATELY ON QUOTA ERROR

        if (
            "429" in error_message
            or "RESOURCE_EXHAUSTED" in error_message
        ):

            print(
                "\n Gemini quota/rate limit reached."
            )

            print(
                "No automatic retry will be performed."
            )

            print(
                "Your existing cached embeddings "
                "are safe."
            )

            return None

        # Other errors
        print(
            "\n Embedding request failed:"
        )

        print(error_message)

        return None

    # SAVE NEW EMBEDDINGS TO CACHE

    for (
        embedding,
        original_index,
        text_hash
    ) in zip(
        new_embeddings,
        original_indices,
        text_hashes
    ):

        # Store result
        results[original_index] = embedding

        # Save to cache
        embedding_cache[text_hash] = embedding

    # Save cache permanently
    save_embedding_cache()

    print(
        f" Generated "
        f"{len(new_embeddings)} new embeddings."
    )

    print(
        f" Cached embeddings: "
        f"{len(embedding_cache)}"
    )

    return results

Loaded 0 cached embeddings.


In [25]:
test_texts = [
    "Revenue increased by 25 percent in Q1 2024.",
    "Consulting Services achieved the fastest growth.",
    "The company launched AI Document Insights."
]

embeddings = get_embeddings_batch(test_texts)

if embeddings is not None:

    print("\n Success!")

    print(
        "Number of embeddings:",
        len(embeddings)
    )

    print(
        "Embedding dimension:",
        len(embeddings[0])
    )

📡 Sending 3 new texts to Gemini...
 Generated 3 new embeddings.
 Cached embeddings: 3

 Success!
Number of embeddings: 3
Embedding dimension: 3072


In [26]:
# Initialize ChromaDB

# INITIALIZE CHROMADB

import chromadb

CHROMA_PATH = str(BASE_DIR / "chroma_db")

chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

COLLECTION_NAME = "documind_documents"

# Delete old collection during development
try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print("Old collection deleted.")
except Exception:
    pass


collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={
        "description": (
            "DocuMind AI multimodal document chunks"
        )
    }
)

print(" ChromaDB collection ready!")
print(f"Collection: {COLLECTION_NAME}")

 ChromaDB collection ready!
Collection: documind_documents


### Prepare chunks for embedding

In [27]:
# PREPARE CHUNKS FOR VECTOR SEARCH

searchable_chunks = [
    chunk
    for chunk in multimodal_chunks
    if chunk["content_type"] in ["text", "table"]
    and chunk["content"].strip()
]

print(
    f"Total searchable chunks: "
    f"{len(searchable_chunks)}"
)

Total searchable chunks: 9


In [28]:
# GENERATE EMBEDDINGS AND STORE IN CHROMADB
# SAFE BATCH + CACHE-FIRST VERSION

from tqdm.auto import tqdm
import time


BATCH_SIZE = 10

successful_chunks = 0
failed_chunks = 0


# Process chunks in batches
for start in tqdm(
    range(0, len(searchable_chunks), BATCH_SIZE),
    desc="Processing batches"
):

    # GET CURRENT BATCH

    batch_chunks = searchable_chunks[
        start : start + BATCH_SIZE
    ]

    batch_texts = [
        chunk["content"]
        for chunk in batch_chunks
    ]


    # GENERATE EMBEDDINGS
    # Uses get_embeddings_batch() instead of old get_embedding()

    embeddings = get_embeddings_batch(
        batch_texts
    )


    # Stop immediately if API quota/rate limit is reached
    if embeddings is None:

        print("\n Embedding stopped due to API quota/rate limit.")

        failed_chunks += len(batch_chunks)

        break


    # PREPARE CHROMADB DATA

    ids = []
    documents = []
    metadatas = []


    for chunk in batch_chunks:

        # Prepare metadata
        metadata = {
            "content_type": chunk["content_type"],
            "document_name": chunk["document_name"],
            "page_number": chunk["page_number"]
        }


        # Add optional metadata
        if "chunk_index" in chunk:

            metadata["chunk_index"] = chunk[
                "chunk_index"
            ]


        if "table_id" in chunk:

            metadata["table_id"] = chunk[
                "table_id"
            ]


        # Add data for this chunk
        ids.append(
            chunk["chunk_id"]
        )

        documents.append(
            chunk["content"]
        )

        metadatas.append(
            metadata
        )


    # STORE ENTIRE BATCH IN CHROMADB

    try:

        collection.upsert(
            ids=ids,
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas
        )

        successful_chunks += len(batch_chunks)

        print(
            f" Stored {successful_chunks} / "
            f"{len(searchable_chunks)} chunks"
        )


    except Exception as e:

        print("\n ChromaDB storage error:")
        print(e)

        failed_chunks += len(batch_chunks)


    # Small delay between batches
    time.sleep(1)


# FINAL RESULTS

print("\n" + "=" * 50)

print("EMBEDDING COMPLETED")

print("=" * 50)

print(
    f"Successfully stored: {successful_chunks}"
)

print(
    f"Failed: {failed_chunks}"
)

print(
    f"Total in ChromaDB: {collection.count()}"
)

Processing batches:   0%|          | 0/1 [00:00<?, ?it/s]

📡 Sending 9 new texts to Gemini...
 Generated 9 new embeddings.
 Cached embeddings: 12
 Stored 9 / 9 chunks

EMBEDDING COMPLETED
Successfully stored: 9
Failed: 0
Total in ChromaDB: 9


In [29]:
# TEST SEMANTIC SEARCH

def semantic_search(query, top_k=5):
    """
    Search relevant document chunks using
    Gemini embeddings + ChromaDB.
    """

    # Generate embedding for the query
    embeddings = get_embeddings_batch([query])

    # Check if embedding generation failed
    if embeddings is None or len(embeddings) == 0:

        print(" Unable to generate query embedding.")

        return None

    # Get the first embedding
    query_embedding = embeddings[0]

    # Check whether ChromaDB contains documents
    total_documents = collection.count()

    if total_documents == 0:

        print(" ChromaDB collection is empty.")

        return None

    # Make sure top_k is not larger than total documents
    n_results = min(top_k, total_documents)

    # Search ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    return results

In [30]:
# Test with a real question
query = "What was the revenue growth in Q1 2024?"

results = semantic_search(
    query=query,
    top_k=5
)

📡 Sending 1 new texts to Gemini...
 Generated 1 new embeddings.
 Cached embeddings: 13


In [31]:
# DISPLAY SEARCH RESULTS

if results is not None:

    print(f"\n Query: {query}\n")

    for i, (
        document,
        metadata,
        distance
    ) in enumerate(
        zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ),
        start=1
    ):

        print("=" * 80)

        print(f"Result #{i}")

        print(
            f"📄 Document: "
            f"{metadata.get('document_name')}"
        )

        print(
            f" Page: "
            f"{metadata.get('page_number')}"
        )

        print(
            f" Type: "
            f"{metadata.get('content_type')}"
        )

        print(
            f" Distance: "
            f"{distance:.4f}"
        )

        print("-" * 80)

        print(document)

        print()


 Query: What was the revenue growth in Q1 2024?

Result #1
📄 Document: documind_sample_report.pdf
 Page: 3
 Type: table
 Distance: 0.4151
--------------------------------------------------------------------------------
Segment | Q1 2024 | Q4 2023 | Change
Software Solutions | $1,320,000 | $1,020,000 | +29.41%
Consulting Services | $720,000 | $540,000 | +33.33%
Support & Maintenance | $410,000 | $400,000 | +2.50%
Total Revenue | $2,450,000 | $1,960,000 | +25.00%

Result #2
📄 Document: documind_sample_report.pdf
 Page: 2
 Type: table
 Distance: 0.4368
--------------------------------------------------------------------------------
Metric | Q1 2024 | Q4 2023 | Change
Revenue | $2,450,000 | $1,960,000 | +25%
Gross Profit | $1,519,000 | $1,166,000 | +30.28%
Gross Profit Margin | 62% | 59% | +3 pp
Operating Expenses | $890,000 | $970,000 | -8.24%
New Customers | 218 | 185 | +17.84%

Result #3
📄 Document: documind_sample_report.pdf
 Page: 4
 Type: table
 Distance: 0.4570
--------------------

### BM25 + Hybrid Retrieval

In [32]:
# Install BM25
!pip install -q rank-bm25

In [33]:
from rank_bm25 import BM25Okapi
import re

In [34]:
# CREATE BM25 INDEX

def tokenize_text(text):
    """
    Simple tokenizer for BM25.
    """

    text = text.lower()

    tokens = re.findall(
        r"\b\w+\b",
        text
    )

    return tokens


# Extract all searchable documents
bm25_documents = [
    chunk["content"]
    for chunk in searchable_chunks
]


# Tokenize documents
tokenized_documents = [
    tokenize_text(document)
    for document in bm25_documents
]


# Create BM25 index
bm25 = BM25Okapi(
    tokenized_documents
)


print(" BM25 index created!")
print(
    "Documents indexed:",
    len(bm25_documents)
)

 BM25 index created!
Documents indexed: 9


In [35]:
# BM25 SEARCH FUNCTION

def bm25_search(query, top_k=5):
    """
    Search documents using BM25 keyword matching.
    """

    query_tokens = tokenize_text(query)

    scores = bm25.get_scores(
        query_tokens
    )

    # Get indices sorted by score
    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    results = []

    for index in top_indices:

        results.append({
            "chunk": searchable_chunks[index],
            "score": float(scores[index])
        })

    return results

In [36]:
query = "revenue growth Q1 2024"

results_bm25 = bm25_search(
    query,
    top_k=5
)

for i, result in enumerate(
    results_bm25,
    start=1
):

    chunk = result["chunk"]

    print("=" * 80)

    print(f"Result #{i}")

    print(
        "Page:",
        chunk["page_number"]
    )

    print(
        "Type:",
        chunk["content_type"]
    )

    print(
        "BM25 Score:",
        round(result["score"], 4)
    )

    print("-" * 80)

    print(
        chunk["content"][:700]
    )

Result #1
Page: 3
Type: text
BM25 Score: 1.8274
--------------------------------------------------------------------------------
2. Financial Performance Revenue grew across all major segments. Software Solutions remained the largest contributor, while Consulting Services experienced the fastest percentage growth. Support and Maintenance continued to provide stable recurring revenue. Segment Q1 2024 Q4 2023 Change Software Solutions $1,320,000 $1,020,000 +29.41% Consulting Services $720,000 $540,000 +33.33% Support & Maintenance $410,000 $400,000 +2.50% Total Revenue $2,450,000 $1,960,000 +25.00% Interpretation The financial data indicates broad-based growth rather than dependence on a single segment. Software Solutions contributed approximately 54 percent of total quarterly revenue. Consulting Services showed the str
Result #2
Page: 5
Type: text
BM25 Score: 1.2457
--------------------------------------------------------------------------------
4. Outlook and Strategic Priorities Manag

In [37]:
# HYBRID RETRIEVAL

def hybrid_search(
    query,
    top_k=5,
    vector_weight=0.6,
    bm25_weight=0.4
):
    """
    Combine Gemini vector search
    with BM25 keyword search.
    """

    # VECTOR SEARCH

    vector_results = semantic_search(
        query=query,
        top_k=top_k * 2
    )

    if vector_results is None:
        print(" Vector search failed.")

        return None


    # BM25 SEARCH

    bm25_results = bm25_search(
        query=query,
        top_k=top_k * 2
    )


    # COMBINE RESULTS

    combined_results = {}


    # Process vector results
    for document, metadata, distance in zip(
        vector_results["documents"][0],
        vector_results["metadatas"][0],
        vector_results["distances"][0]
    ):

        # Convert distance to similarity
        similarity = 1 / (1 + distance)

        key = document


        combined_results[key] = {
            "content": document,
            "metadata": metadata,
            "vector_score": similarity,
            "bm25_score": 0.0
        }


    # Process BM25 results
    for result in bm25_results:

        chunk = result["chunk"]

        key = chunk["content"]


        if key not in combined_results:

            combined_results[key] = {
                "content": chunk["content"],

                "metadata": {
                    "content_type":
                        chunk["content_type"],

                    "document_name":
                        chunk["document_name"],

                    "page_number":
                        chunk["page_number"]
                },

                "vector_score": 0.0,

                "bm25_score":
                    result["score"]
            }

        else:

            combined_results[key][
                "bm25_score"
            ] = result["score"]


    # NORMALIZE SCORES

    max_vector_score = max(
        [
            item["vector_score"]
            for item in combined_results.values()
        ],
        default=1
    )


    max_bm25_score = max(
        [
            item["bm25_score"]
            for item in combined_results.values()
        ],
        default=1
    )


    # Calculate final hybrid score
    for item in combined_results.values():

        normalized_vector = (
            item["vector_score"]
            / max_vector_score
            if max_vector_score > 0
            else 0
        )


        normalized_bm25 = (
            item["bm25_score"]
            / max_bm25_score
            if max_bm25_score > 0
            else 0
        )


        item["hybrid_score"] = (

            vector_weight
            * normalized_vector

            +

            bm25_weight
            * normalized_bm25
        )


    # SORT RESULTS

    final_results = sorted(

        combined_results.values(),

        key=lambda x: x["hybrid_score"],

        reverse=True

    )[:top_k]


    return final_results

In [38]:
query = "Which business segment had the fastest growth?"

hybrid_results = hybrid_search(
    query=query,
    top_k=5
)

📡 Sending 1 new texts to Gemini...
 Generated 1 new embeddings.
 Cached embeddings: 14


In [39]:
# DISPLAY HYBRID RESULTS

if hybrid_results:

    print(f"\n Query: {query}\n")

    for i, result in enumerate(
        hybrid_results,
        start=1
    ):

        print("=" * 80)

        print(
            f"Result #{i}"
        )

        print(
            "Page:",
            result["metadata"].get(
                "page_number"
            )
        )

        print(
            "Type:",
            result["metadata"].get(
                "content_type"
            )
        )

        print(
            "Vector Score:",
            round(
                result["vector_score"],
                4
            )
        )

        print(
            "BM25 Score:",
            round(
                result["bm25_score"],
                4
            )
        )

        print(
            "Hybrid Score:",
            round(
                result["hybrid_score"],
                4
            )
        )

        print("-" * 80)

        print(
            result["content"][:800]
        )

        print()


 Query: Which business segment had the fastest growth?

Result #1
Page: 3
Type: text
Vector Score: 0.6185
BM25 Score: 2.8192
Hybrid Score: 0.9898
--------------------------------------------------------------------------------
2. Financial Performance Revenue grew across all major segments. Software Solutions remained the largest contributor, while Consulting Services experienced the fastest percentage growth. Support and Maintenance continued to provide stable recurring revenue. Segment Q1 2024 Q4 2023 Change Software Solutions $1,320,000 $1,020,000 +29.41% Consulting Services $720,000 $540,000 +33.33% Support & Maintenance $410,000 $400,000 +2.50% Total Revenue $2,450,000 $1,960,000 +25.00% Interpretation The financial data indicates broad-based growth rather than dependence on a single segment. Software Solutions contributed approximately 54 percent of total quarterly revenue. Consulting Services showed the strongest growth rate at 33.33 percent.

Result #2
Page: 5
Type: text
Vecto

### Gemini Answer Generation with Source Citations

In [40]:
# GEMINI GENERATION MODEL

GENERATION_MODEL = "gemini-3.6-flash"

print("Generation model:", GENERATION_MODEL)

Generation model: gemini-3.6-flash


In [41]:
# Create Context from Hybrid Results
# BUILD RAG CONTEXT

def build_context(hybrid_results):
    """
    Convert retrieved chunks into structured context
    for the Gemini model.
    """

    context_parts = []

    for i, result in enumerate(hybrid_results, start=1):

        metadata = result["metadata"]

        document_name = metadata.get(
            "document_name",
            "Unknown Document"
        )

        page_number = metadata.get(
            "page_number",
            "Unknown"
        )

        content_type = metadata.get(
            "content_type",
            "text"
        )

        context_part = f"""
SOURCE {i}

Document: {document_name}
Page: {page_number}
Content Type: {content_type}

Content:
{result["content"]}
"""

        context_parts.append(context_part)

    return "\n\n".join(context_parts)

In [42]:
# Create the RAG Answer Function
# RAG ANSWER GENERATION

def generate_rag_answer(
    query,
    top_k=5
):
    """
    Complete RAG pipeline:

    User Query
        ↓
    Hybrid Retrieval
        ↓
    Context Construction
        ↓
    Gemini Generation
        ↓
    Answer with Page Citations
    """

    print(" Searching documents...")

    # STEP 1: HYBRID RETRIEVAL

    retrieved_results = hybrid_search(
        query=query,
        top_k=top_k
    )

    if not retrieved_results:

        return {
            "answer": "I could not find relevant information in the document.",
            "sources": []
        }


    # STEP 2: BUILD CONTEXT

    context = build_context(
        retrieved_results
    )


    # STEP 3: CREATE PROMPT

    prompt = f"""
You are DocuMind AI, a Multimodal Document Intelligence assistant.

Your job is to answer questions using ONLY the document context provided below.

IMPORTANT RULES:

1. Use only the provided context.
2. Do not use outside knowledge.
3. If the answer is not available in the context, say:
   "I could not find this information in the provided document."
4. Give a clear and concise answer.
5. Support factual statements with source citations.
6. Use this citation format exactly:

[Document Name | Page X]

DOCUMENT CONTEXT:

{context}

USER QUESTION:

{query}

ANSWER:
"""


    # STEP 4: GEMINI GENERATION

    try:

        response = client.models.generate_content(
            model=GENERATION_MODEL,
            contents=prompt
        )

        answer = response.text


    except Exception as e:

        print(" Gemini generation error:")
        print(e)

        return {
            "answer": None,
            "sources": []
        }


    # STEP 5: PREPARE SOURCE INFORMATION

    sources = []

    seen_sources = set()

    for result in retrieved_results:

        metadata = result["metadata"]

        source_key = (
            metadata.get("document_name"),
            metadata.get("page_number")
        )

        if source_key not in seen_sources:

            sources.append({
                "document_name":
                    metadata.get(
                        "document_name"
                    ),

                "page_number":
                    metadata.get(
                        "page_number"
                    ),

                "content_type":
                    metadata.get(
                        "content_type"
                    )
            })

            seen_sources.add(
                source_key
            )


    return {
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": retrieved_results
    }

In [43]:
# Test Complete RAG Pipeline
# TEST DOCUMIND AI

query = "What was the revenue growth in Q1 2024?"

response = generate_rag_answer(
    query=query,
    top_k=5
)

print("\n" + "=" * 80)
print(" DOCUMIND AI ANSWER")
print("=" * 80)

print("\n" + response["answer"])

print("\n SOURCES")

for source in response["sources"]:

    print(
        f"- {source['document_name']} "
        f"| Page {source['page_number']}"
    )

 Searching documents...
 All embeddings loaded from cache.

 DOCUMIND AI ANSWER

In Q1 2024, revenue increased by 25% year-over-year, reaching $2.45 million (up from $1,960,000 in Q4 2023) [documind_sample_report.pdf | Page 2] [documind_sample_report.pdf | Page 3] [documind_sample_report.pdf | Page 5].

 SOURCES
- documind_sample_report.pdf | Page 5
- documind_sample_report.pdf | Page 3
- documind_sample_report.pdf | Page 4
- documind_sample_report.pdf | Page 2


In [44]:
# Test Multiple Questions
test_queries = [
    "What was the revenue growth in Q1 2024?",
    "Which business segment had the fastest growth?",
    "How many new customers were acquired?",
    "What product features were launched?",
    "What are the strategic priorities?"
]

In [45]:
for query in test_queries:

    print("\n" + "=" * 100)
    print("QUESTION:", query)
    print("=" * 100)

    response = generate_rag_answer(
        query=query,
        top_k=5
    )

    print("\nANSWER:\n")

    print(response["answer"])

    print("\nSOURCES:")

    for source in response["sources"]:

        print(
            f"- {source['document_name']} "
            f"| Page {source['page_number']}"
        )


QUESTION: What was the revenue growth in Q1 2024?
 Searching documents...
 All embeddings loaded from cache.

ANSWER:

In Q1 2024, revenue grew by 25% (or 25.00%) to $2.45 million ($2,450,000), compared to $1.96 million ($1,960,000) in Q4 2023 [documind_sample_report.pdf | Page 2] [documind_sample_report.pdf | Page 3] [documind_sample_report.pdf | Page 5].

SOURCES:
- documind_sample_report.pdf | Page 5
- documind_sample_report.pdf | Page 3
- documind_sample_report.pdf | Page 4
- documind_sample_report.pdf | Page 2

QUESTION: Which business segment had the fastest growth?
 Searching documents...
 All embeddings loaded from cache.

ANSWER:

Consulting Services had the fastest growth, achieving a growth rate of 33.33% [documind_sample_report.pdf | Page 3].

SOURCES:
- documind_sample_report.pdf | Page 3
- documind_sample_report.pdf | Page 5
- documind_sample_report.pdf | Page 1
- documind_sample_report.pdf | Page 4

QUESTION: How many new customers were acquired?
 Searching documents...

### True Multimodal RAG: Image & Chart Understanding

### Image Extraction

### Extract embedded images from PDF documents and save them locally.

### Each image is linked to its source document and page number.

In [46]:
# EXTRACT IMAGES FROM PDF

import pymupdf
from pathlib import Path


# Create image directory
IMAGE_DIR = BASE_DIR / "data" / "images"

IMAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def extract_images_from_pdf(pdf_path):
    """
    Extract embedded images from a PDF.

    Returns image metadata including:
    - image_id
    - image_path
    - document_name
    - page_number
    """

    pdf_path = Path(pdf_path)

    document = pymupdf.open(pdf_path)

    extracted_images = []

    seen_xrefs = set()


    for page_index in range(len(document)):

        page = document[page_index]

        image_list = page.get_images(
            full=True
        )


        for image_index, image_info in enumerate(
            image_list
        ):

            xref = image_info[0]


            # Skip duplicate images
            if xref in seen_xrefs:
                continue

            seen_xrefs.add(xref)


            try:

                image_data = document.extract_image(
                    xref
                )

                image_bytes = image_data["image"]

                image_extension = image_data["ext"]


                image_filename = (
                    f"{pdf_path.stem}"
                    f"_page_{page_index + 1}"
                    f"_image_{image_index + 1}"
                    f".{image_extension}"
                )


                image_path = (
                    IMAGE_DIR / image_filename
                )


                with open(
                    image_path,
                    "wb"
                ) as file:

                    file.write(image_bytes)


                extracted_images.append({

                    "image_id":
                        f"{pdf_path.stem}"
                        f"_page_{page_index + 1}"
                        f"_image_{image_index + 1}",

                    "image_path":
                        str(image_path),

                    "document_name":
                        pdf_path.name,

                    "page_number":
                        page_index + 1

                })


            except Exception as e:

                print(
                    f" Could not extract image "
                    f"from page {page_index + 1}: {e}"
                )


    document.close()

    return extracted_images

In [47]:
# RUN IMAGE EXTRACTION

extracted_images = extract_images_from_pdf(
    PDF_PATH
)

print(
    f"\n Extracted "
    f"{len(extracted_images)} images"
)

for image in extracted_images:

    print(
        f"\nImage ID: {image['image_id']}"
    )

    print(
        f"Page: {image['page_number']}"
    )

    print(
        f"Path: {image['image_path']}"
    )


 Extracted 0 images


In [48]:
# RENDER PDF PAGES AS IMAGES

PAGE_IMAGE_DIR = BASE_DIR / "data" / "page_images"

PAGE_IMAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def render_pdf_pages(pdf_path, zoom=1.5):
    """
    Convert each PDF page into a PNG image.
    """

    pdf_path = Path(pdf_path)

    document = pymupdf.open(pdf_path)

    rendered_pages = []


    for page_index in tqdm(
        range(len(document)),
        desc="Rendering PDF pages"
    ):

        page = document[page_index]

        matrix = pymupdf.Matrix(
            zoom,
            zoom
        )

        pixmap = page.get_pixmap(
            matrix=matrix,
            alpha=False
        )


        image_path = (
            PAGE_IMAGE_DIR
            / f"{pdf_path.stem}_page_{page_index + 1}.png"
        )


        pixmap.save(
            str(image_path)
        )


        rendered_pages.append({

            "image_id":
                f"{pdf_path.stem}_page_{page_index + 1}",

            "image_path":
                str(image_path),

            "document_name":
                pdf_path.name,

            "page_number":
                page_index + 1

        })


    document.close()

    return rendered_pages

In [49]:
# RENDER ALL PAGES

page_images = render_pdf_pages(
    PDF_PATH
)

print(
    f"\n Rendered "
    f"{len(page_images)} PDF pages"
)

print(page_images[:2])

Rendering PDF pages:   0%|          | 0/5 [00:00<?, ?it/s]


 Rendered 5 PDF pages
[{'image_id': 'documind_sample_report_page_1', 'image_path': '/kaggle/working/documind-ai/data/page_images/documind_sample_report_page_1.png', 'document_name': 'documind_sample_report.pdf', 'page_number': 1}, {'image_id': 'documind_sample_report_page_2', 'image_path': '/kaggle/working/documind-ai/data/page_images/documind_sample_report_page_2.png', 'document_name': 'documind_sample_report.pdf', 'page_number': 2}]


In [50]:
# Create Image Search
# GET RELEVANT PAGE IMAGES

def get_relevant_page_images(
    retrieved_results,
    page_images,
    max_images=3
):
    """
    Select PDF page images based on
    pages retrieved by hybrid search.
    """

    relevant_pages = []

    seen_pages = set()


    for result in retrieved_results:

        page_number = result["metadata"].get(
            "page_number"
        )

        if page_number not in seen_pages:

            relevant_pages.append(
                page_number
            )

            seen_pages.add(
                page_number
            )

        if len(relevant_pages) >= max_images:

            break


    # Match page numbers to images
    relevant_images = []

    for page_number in relevant_pages:

        for image_info in page_images:

            if (
                image_info["page_number"]
                == page_number
            ):

                relevant_images.append(
                    image_info
                )

                break


    return relevant_images

In [51]:
# Upgrade RAG Answer to Multimodal
# MULTIMODAL RAG ANSWER GENERATION

from PIL import Image


def generate_multimodal_rag_answer(
    query,
    top_k=5,
    max_images=3
):
    """
    Complete Multimodal RAG pipeline:

    Query
      ↓
    Hybrid Search
      ↓
    Text Context
      +
    Relevant PDF Page Images
      ↓
    Gemini Multimodal Analysis
      ↓
    Answer + Page Citations
    """

    print(" Searching documents...")


    # STEP 1: HYBRID RETRIEVAL

    retrieved_results = hybrid_search(
        query=query,
        top_k=top_k
    )


    if not retrieved_results:

        return {
            "answer":
                "I could not find relevant information "
                "in the document.",
            "sources": []
        }


    # STEP 2: BUILD TEXT CONTEXT

    context = build_context(
        retrieved_results
    )


    # STEP 3: GET RELEVANT PAGE IMAGES

    relevant_images = get_relevant_page_images(
        retrieved_results=retrieved_results,
        page_images=page_images,
        max_images=max_images
    )


    # STEP 4: BUILD MULTIMODAL PROMPT

    prompt = f"""
You are DocuMind AI, a Multimodal Document Intelligence assistant.

Answer the user's question using ONLY:

1. The retrieved document text context.
2. The PDF page images provided.

Do not use outside knowledge.

If the answer cannot be found, say:

"I could not find this information in the provided document."

For factual statements, include citations using:

[Document Name | Page X]

RETRIEVED TEXT CONTEXT:

{context}

USER QUESTION:

{query}

ANSWER:
"""


    # STEP 5: LOAD RELEVANT IMAGES

    image_parts = []

    for image_info in relevant_images:

        image = Image.open(
            image_info["image_path"]
        )

        image_parts.append(
            image.copy()
        )

        image.close()


    # STEP 6: GEMINI MULTIMODAL GENERATION

    try:

        contents = [
            prompt,
            *image_parts
        ]


        response = client.models.generate_content(
            model=GENERATION_MODEL,
            contents=contents
        )

        answer = response.text


    except Exception as e:

        print(
            " Multimodal generation error:"
        )

        print(e)

        return {
            "answer": None,
            "sources": []
        }


    # ----------------------------------------
    # STEP 7: PREPARE SOURCES
    # ----------------------------------------

    sources = []

    seen_sources = set()


    for result in retrieved_results:

        metadata = result["metadata"]

        source_key = (
            metadata.get("document_name"),
            metadata.get("page_number")
        )


        if source_key not in seen_sources:

            sources.append({

                "document_name":
                    metadata.get(
                        "document_name"
                    ),

                "page_number":
                    metadata.get(
                        "page_number"
                    )

            })

            seen_sources.add(
                source_key
            )


    return {

        "answer":
            answer,

        "sources":
            sources,

        "retrieved_chunks":
            retrieved_results,

        "images_used":
            relevant_images
    }

In [52]:
# TEST MULTIMODAL DOCUMIND AI

query = "What information is shown in the revenue chart or financial results?"

response = generate_multimodal_rag_answer(
    query=query,
    top_k=5,
    max_images=3
)

print("\n" + "=" * 80)

print(" DOCUMIND AI — MULTIMODAL ANSWER")

print("=" * 80)

print("\nANSWER:\n")

print(response["answer"])


print("\n SOURCES:")

for source in response["sources"]:

    print(
        f"- {source['document_name']} "
        f"| Page {source['page_number']}"
    )


print("\n IMAGES USED:")

for image in response.get(
    "images_used",
    []
):

    print(
        f"- Page {image['page_number']}: "
        f"{image['image_path']}"
    )

 Searching documents...
📡 Sending 1 new texts to Gemini...
 Generated 1 new embeddings.
 Cached embeddings: 18
 Multimodal generation error:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

 DOCUMIND AI — MULTIMODAL ANSWER

ANSWER:

None

 SOURCES:

 IMAGES USED:


### Build the Gradio UI

In [53]:
# Install Gradio
!pip install -q gradio

In [54]:
# create a simple UI for the current pipeline
import gradio as gr

def ask_documind(question):
    """
    Process a user question using
    the existing Multimodal RAG pipeline.
    """

    if not question or not question.strip():
        return "Please enter a question.", ""

    try:
        response = generate_multimodal_rag_answer(
            query=question,
            top_k=5,
            max_images=3
        )

        answer = response.get("answer")

        if not answer:
            return "Unable to generate an answer.", ""

        # Format sources
        sources = response.get("sources", [])

        source_text = ""

        if sources:

            source_text = "### Sources\n\n"

            for source in sources:

                source_text += (
                    f"- **{source['document_name']}** "
                    f"| Page {source['page_number']}\n"
                )

        else:

            source_text = "No sources found."

        return answer, source_text

    except Exception as e:

        return (
            f" Error: {str(e)}",
            ""
        )

In [56]:
# Create the Gradio Interface
with gr.Blocks(
    title="DocuMind AI"
) as demo:

    gr.Markdown("""
    #  DocuMind AI

    ### Multimodal Document Intelligence RAG

    Ask questions about your document using:

    -  Hybrid Retrieval
    -  Gemini AI
    -  Table and chart understanding
    -  Multimodal page analysis
    -  Page-level source citations
    """)

    with gr.Row():

        with gr.Column():

            question_input = gr.Textbox(
                label="Ask a question",
                placeholder=(
                    "Example: What was the revenue "
                    "growth in Q1 2024?"
                ),
                lines=3
            )

            ask_button = gr.Button(
                "Ask DocuMind AI"
            )

        with gr.Column():

            answer_output = gr.Markdown(
                label="Answer"
            )

            sources_output = gr.Markdown(
                label="Sources"
            )


    gr.Examples(
        examples=[
            ["What was the revenue growth in Q1 2024?"],
            ["Which business segment had the fastest growth?"],
            ["How many new customers were acquired?"],
            ["What product features were launched?"],
            ["What are the strategic priorities?"]
        ],
        inputs=question_input
    )


    ask_button.click(
        fn=ask_documind,
        inputs=question_input,
        outputs=[
            answer_output,
            sources_output
        ]
    )


demo.launch(
    share=True,
    debug=True
)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://57fa7c235bc5cee660.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://57fa7c235bc5cee660.gradio.live


In [57]:
# Add PDF Upload and Process Any Document
# PROCESS UPLOADED PDF

import pymupdf
import uuid
import shutil
from pathlib import Path
from tqdm.auto import tqdm


# Store uploaded PDFs
APP_UPLOAD_DIR = BASE_DIR / "data" / "app_uploads"

APP_UPLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def extract_text_from_uploaded_pdf(pdf_path):
    """
    Extract text from an uploaded PDF.
    """

    document = pymupdf.open(pdf_path)

    extracted_pages = []

    for page_number, page in enumerate(
        document,
        start=1
    ):

        text = page.get_text("text")

        if text and text.strip():

            extracted_pages.append({
                "page_number": page_number,
                "content": text.strip()
            })

    document.close()

    return extracted_pages

In [58]:
# CHUNK UPLOADED DOCUMENT

def chunk_uploaded_document(
    pages,
    document_name,
    chunk_size=1000,
    chunk_overlap=150
):
    """
    Split extracted PDF text into overlapping chunks.
    """

    chunks = []

    for page_data in pages:

        text = page_data["content"]

        page_number = page_data["page_number"]

        start = 0
        chunk_index = 0

        while start < len(text):

            end = start + chunk_size

            chunk_text = text[start:end].strip()

            if chunk_text:

                chunks.append({

                    "chunk_id":
                        f"{document_name}_"
                        f"page_{page_number}_"
                        f"chunk_{chunk_index}_"
                        f"{uuid.uuid4().hex[:8]}",

                    "content":
                        chunk_text,

                    "content_type":
                        "text",

                    "document_name":
                        document_name,

                    "page_number":
                        page_number,

                    "chunk_index":
                        chunk_index
                })

            start += (
                chunk_size - chunk_overlap
            )

            chunk_index += 1

    return chunks

In [59]:
# CREATE COLLECTION FOR UPLOADED DOCUMENT

import chromadb


def create_document_collection(document_id):
    """
    Create a separate ChromaDB collection
    for each uploaded document.
    """

    collection_name = (
        f"documind_"
        f"{document_id}"
    )

    # Remove old collection if it exists
    try:

        chroma_client.delete_collection(
            name=collection_name
        )

    except:
        pass

    collection = chroma_client.create_collection(
        name=collection_name
    )

    return collection

In [60]:
# MAIN DOCUMENT PROCESSING PIPELINE

def process_uploaded_pdf(pdf_file):
    """
    Complete processing pipeline for an uploaded PDF.

    Returns:
        status message
        document state
    """

    global active_collection
    global active_searchable_chunks
    global active_bm25
    global active_page_images
    global active_document_name


    if pdf_file is None:

        return (
            " Please upload a PDF first.",
            None
        )


    try:

        # STEP 1: GET FILE INFORMATION

        source_path = Path(pdf_file)

        document_name = source_path.name

        document_id = uuid.uuid4().hex[:12]


        # STEP 2: COPY PDF TO APP DIRECTORY

        destination_path = (
            APP_UPLOAD_DIR /
            f"{document_id}_{document_name}"
        )

        shutil.copy(
            source_path,
            destination_path
        )


        # STEP 3: EXTRACT TEXT

        pages = extract_text_from_uploaded_pdf(
            destination_path
        )

        if not pages:

            return (
                " No readable text found in this PDF.",
                None
            )


        # STEP 4: CREATE CHUNKS

        chunks = chunk_uploaded_document(
            pages=pages,
            document_name=document_name
        )


        if not chunks:

            return (
                " No chunks could be created.",
                None
            )


        # STEP 5: CREATE CHROMADB COLLECTION

        document_collection = (
            create_document_collection(
                document_id
            )
        )


        # STEP 6: CREATE EMBEDDINGS IN BATCHES

        BATCH_SIZE = 10

        processed_count = 0


        for start in range(
            0,
            len(chunks),
            BATCH_SIZE
        ):

            batch_chunks = chunks[
                start:start + BATCH_SIZE
            ]

            batch_texts = [
                chunk["content"]
                for chunk in batch_chunks
            ]


            embeddings = get_embeddings_batch(
                batch_texts
            )


            # Stop safely if quota reached
            if embeddings is None:

                return (
                    f" Processing stopped at "
                    f"{processed_count}/{len(chunks)} chunks "
                    f"because the Gemini embedding quota "
                    f"was reached.",
                    None
                )


            ids = []
            documents = []
            metadatas = []


            for chunk in batch_chunks:

                ids.append(
                    chunk["chunk_id"]
                )

                documents.append(
                    chunk["content"]
                )

                metadatas.append({

                    "content_type":
                        chunk["content_type"],

                    "document_name":
                        chunk["document_name"],

                    "page_number":
                        chunk["page_number"],

                    "chunk_index":
                        chunk["chunk_index"]
                })


            # Store embeddings
            document_collection.upsert(
                ids=ids,
                documents=documents,
                embeddings=embeddings,
                metadatas=metadatas
            )


            processed_count += len(
                batch_chunks
            )


        # STEP 7: BUILD BM25 INDEX

        bm25_documents = [
            chunk["content"]
            for chunk in chunks
        ]

        tokenized_documents = [
            tokenize_text(document)
            for document in bm25_documents
        ]

        document_bm25 = BM25Okapi(
            tokenized_documents
        )


        # STEP 8: RENDER PDF PAGES

        document_page_dir = (
            PAGE_IMAGE_DIR /
            document_id
        )

        document_page_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        pdf_document = pymupdf.open(
            destination_path
        )

        document_page_images = []


        for page_index in range(
            len(pdf_document)
        ):

            page = pdf_document[
                page_index
            ]

            pixmap = page.get_pixmap(
                matrix=pymupdf.Matrix(
                    1.5,
                    1.5
                ),
                alpha=False
            )


            image_path = (
                document_page_dir /
                f"page_{page_index + 1}.png"
            )


            pixmap.save(
                str(image_path)
            )


            document_page_images.append({

                "image_id":
                    f"{document_id}_"
                    f"page_{page_index + 1}",

                "image_path":
                    str(image_path),

                "document_name":
                    document_name,

                "page_number":
                    page_index + 1
            })


        pdf_document.close()


        # STEP 9: SAVE ACTIVE DOCUMENT STATE

        active_collection = (
            document_collection
        )

        active_searchable_chunks = chunks

        active_bm25 = document_bm25

        active_page_images = (
            document_page_images
        )

        active_document_name = (
            document_name
        )


        return (

            f"""
 Document processed successfully!

 Document: {document_name}

 Pages processed: {len(pages)}

 Chunks created: {len(chunks)}

 Embeddings stored: {processed_count}

 BM25 index: Ready

 Page images: {len(document_page_images)}

You can now ask questions about this PDF.
""",

            document_name
        )


    except Exception as e:

        return (
            f" Processing error:\n\n{str(e)}",
            None
        )

In [61]:
# ACTIVE DOCUMENT VECTOR SEARCH

def active_semantic_search(
    query,
    top_k=5
):

    global active_collection

    if active_collection is None:

        return None


    embeddings = get_embeddings_batch(
        [query]
    )


    if embeddings is None:

        return None


    query_embedding = embeddings[0]

    total_documents = (
        active_collection.count()
    )


    if total_documents == 0:

        return None


    results = active_collection.query(
        query_embeddings=[
            query_embedding
        ],

        n_results=min(
            top_k,
            total_documents
        ),

        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )


    return results

In [62]:
# ACTIVE DOCUMENT BM25 SEARCH

def active_bm25_search(
    query,
    top_k=5
):

    global active_bm25
    global active_searchable_chunks


    if active_bm25 is None:

        return []


    query_tokens = tokenize_text(
        query
    )


    scores = active_bm25.get_scores(
        query_tokens
    )


    top_indices = sorted(

        range(len(scores)),

        key=lambda i: scores[i],

        reverse=True

    )[:top_k]


    results = []


    for index in top_indices:

        results.append({

            "chunk":
                active_searchable_chunks[index],

            "score":
                float(scores[index])
        })


    return results

In [63]:
# ACTIVE HYBRID SEARCH

def active_hybrid_search(
    query,
    top_k=5,
    vector_weight=0.6,
    bm25_weight=0.4
):

    vector_results = (
        active_semantic_search(
            query,
            top_k=top_k * 2
        )
    )


    if vector_results is None:

        return None


    bm25_results = (
        active_bm25_search(
            query,
            top_k=top_k * 2
        )
    )


    combined_results = {}


    # Vector results
    for document, metadata, distance in zip(

        vector_results["documents"][0],

        vector_results["metadatas"][0],

        vector_results["distances"][0]
    ):

        similarity = 1 / (
            1 + distance
        )

        combined_results[document] = {

            "content": document,

            "metadata": metadata,

            "vector_score": similarity,

            "bm25_score": 0.0
        }


    # BM25 results
    for result in bm25_results:

        chunk = result["chunk"]

        key = chunk["content"]


        if key not in combined_results:

            combined_results[key] = {

                "content":
                    chunk["content"],

                "metadata": {

                    "content_type":
                        chunk["content_type"],

                    "document_name":
                        chunk["document_name"],

                    "page_number":
                        chunk["page_number"]
                },

                "vector_score": 0.0,

                "bm25_score":
                    result["score"]
            }

        else:

            combined_results[key][
                "bm25_score"
            ] = result["score"]


    # Normalize scores
    max_vector = max(

        [
            item["vector_score"]
            for item in combined_results.values()
        ],

        default=1
    )


    max_bm25 = max(

        [
            item["bm25_score"]
            for item in combined_results.values()
        ],

        default=1
    )


    for item in combined_results.values():

        normalized_vector = (

            item["vector_score"]
            / max_vector

            if max_vector > 0

            else 0
        )


        normalized_bm25 = (

            item["bm25_score"]
            / max_bm25

            if max_bm25 > 0

            else 0
        )


        item["hybrid_score"] = (

            vector_weight
            * normalized_vector

            +

            bm25_weight
            * normalized_bm25
        )


    final_results = sorted(

        combined_results.values(),

        key=lambda x:
            x["hybrid_score"],

        reverse=True

    )[:top_k]


    return final_results

In [64]:
# ANSWER QUESTIONS ABOUT ACTIVE DOCUMENT

def ask_uploaded_document(
    query,
    top_k=5,
    max_images=3
):

    global active_page_images


    retrieved_results = (
        active_hybrid_search(
            query=query,
            top_k=top_k
        )
    )


    if not retrieved_results:

        return {
            "answer":
                "I could not find relevant information "
                "in the uploaded document.",

            "sources": []
        }


    # Build text context
    context = build_context(
        retrieved_results
    )


    # Get relevant page images
    relevant_images = (
        get_relevant_page_images(

            retrieved_results,

            active_page_images,

            max_images
        )
    )


    prompt = f"""
You are DocuMind AI.

Answer the user's question using ONLY:

1. Retrieved text from the uploaded document.
2. The provided PDF page images.

Do not use outside knowledge.

If the information is unavailable, say:

"I could not find this information in the uploaded document."

Include factual source citations in this format:

[Document Name | Page X]

DOCUMENT CONTEXT:

{context}

QUESTION:

{query}

ANSWER:
"""


    try:

        image_parts = []

        for image_info in relevant_images:

            image = Image.open(
                image_info["image_path"]
            )

            image_parts.append(
                image.copy()
            )

            image.close()


        response = client.models.generate_content(

            model=GENERATION_MODEL,

            contents=[
                prompt,
                *image_parts
            ]
        )


        answer = response.text


    except Exception as e:

        return {

            "answer":
                f" Gemini error: {str(e)}",

            "sources": []
        }


    # Prepare unique sources
    sources = []

    seen = set()


    for result in retrieved_results:

        metadata = result["metadata"]

        key = (

            metadata.get(
                "document_name"
            ),

            metadata.get(
                "page_number"
            )
        )


        if key not in seen:

            sources.append({

                "document_name":
                    key[0],

                "page_number":
                    key[1]
            })

            seen.add(key)


    return {

        "answer": answer,

        "sources": sources
    }

In [65]:
# INITIALIZE ACTIVE DOCUMENT STATE

active_collection = None
active_searchable_chunks = None
active_bm25 = None
active_page_images = None
active_document_name = None

print(" Active document system initialized.")

 Active document system initialized.


In [66]:
# IMPROVED DOCUMENT PROCESSING

import time
import traceback
import gradio as gr


def process_uploaded_pdf(pdf_file, progress=gr.Progress()):

    global active_collection
    global active_searchable_chunks
    global active_bm25
    global active_page_images
    global active_document_name

    if pdf_file is None:
        return (
            " Please upload a PDF first.",
            None
        )

    try:

        progress(0.05, desc="Checking uploaded PDF...")

        source_path = Path(pdf_file)

        if not source_path.exists():
            return (
                " Uploaded file could not be found.",
                None
            )

        if source_path.suffix.lower() != ".pdf":
            return (
                " Please upload a valid PDF file.",
                None
            )

        document_name = source_path.name
        document_id = uuid.uuid4().hex[:12]

        # COPY FILE

        progress(0.10, desc="Preparing document...")

        destination_path = (
            APP_UPLOAD_DIR /
            f"{document_id}_{document_name}"
        )

        shutil.copy(
            source_path,
            destination_path
        )

        # EXTRACT TEXT

        progress(0.20, desc="Extracting text from PDF...")

        pages = extract_text_from_uploaded_pdf(
            destination_path
        )

        if not pages:

            return (
                " No readable text found in this PDF.\n\n"
                "This may be a scanned PDF. OCR support "
                "can be added later.",
                None
            )

        # CREATE CHUNKS

        progress(0.30, desc="Creating document chunks...")

        chunks = chunk_uploaded_document(
            pages=pages,
            document_name=document_name
        )

        if not chunks:

            return (
                " No searchable chunks could be created.",
                None
            )

        # CREATE COLLECTION

        progress(0.35, desc="Creating vector database...")

        document_collection = create_document_collection(
            document_id
        )

        # CREATE EMBEDDINGS

        BATCH_SIZE = 10

        processed_count = 0
        total_batches = (
            len(chunks) + BATCH_SIZE - 1
        ) // BATCH_SIZE

        for batch_number, start in enumerate(
            range(0, len(chunks), BATCH_SIZE),
            start=1
        ):

            batch_chunks = chunks[
                start:start + BATCH_SIZE
            ]

            batch_texts = [
                chunk["content"]
                for chunk in batch_chunks
            ]

            # Dynamic progress from 35% to 75%
            embedding_progress = (
                0.35 +
                (batch_number / total_batches) * 0.40
            )

            progress(
                embedding_progress,
                desc=(
                    f"Creating embeddings "
                    f"({processed_count}/{len(chunks)} chunks)..."
                )
            )

            embeddings = get_embeddings_batch(
                batch_texts
            )

            if embeddings is None:

                return (
                    " Gemini embedding quota reached.\n\n"
                    f"Processed {processed_count} of "
                    f"{len(chunks)} chunks.\n\n"
                    "Please wait for your quota to reset "
                    "and try again.",
                    None
                )

            ids = [
                chunk["chunk_id"]
                for chunk in batch_chunks
            ]

            documents = [
                chunk["content"]
                for chunk in batch_chunks
            ]

            metadatas = []

            for chunk in batch_chunks:

                metadatas.append({

                    "content_type":
                        chunk["content_type"],

                    "document_name":
                        chunk["document_name"],

                    "page_number":
                        chunk["page_number"],

                    "chunk_index":
                        chunk["chunk_index"]
                })

            document_collection.upsert(
                ids=ids,
                documents=documents,
                embeddings=embeddings,
                metadatas=metadatas
            )

            processed_count += len(batch_chunks)

        # BUILD BM25

        progress(0.80, desc="Building keyword search index...")

        bm25_documents = [
            chunk["content"]
            for chunk in chunks
        ]

        tokenized_documents = [
            tokenize_text(text)
            for text in bm25_documents
        ]

        document_bm25 = BM25Okapi(
            tokenized_documents
        )

        # RENDER PAGES

        progress(0.90, desc="Preparing document pages...")

        document_page_dir = (
            PAGE_IMAGE_DIR / document_id
        )

        document_page_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        pdf_document = pymupdf.open(
            destination_path
        )

        document_page_images = []

        total_pdf_pages = len(pdf_document)

        for page_index in range(total_pdf_pages):

            page = pdf_document[page_index]

            pixmap = page.get_pixmap(
                matrix=pymupdf.Matrix(1.5, 1.5),
                alpha=False
            )

            image_path = (
                document_page_dir /
                f"page_{page_index + 1}.png"
            )

            pixmap.save(str(image_path))

            document_page_images.append({

                "image_id":
                    f"{document_id}_page_{page_index + 1}",

                "image_path":
                    str(image_path),

                "document_name":
                    document_name,

                "page_number":
                    page_index + 1
            })

        pdf_document.close()

        # SAVE ACTIVE STATE ONLY AFTER SUCCESS

        progress(0.98, desc="Finalizing document...")

        active_collection = document_collection
        active_searchable_chunks = chunks
        active_bm25 = document_bm25
        active_page_images = document_page_images
        active_document_name = document_name

        progress(1.0, desc="Document ready!")

        return (
            f"""
#  Document Ready

** Document:** {document_name}

-  Pages: {len(pages)}
-  Chunks: {len(chunks)}
-  Embeddings: {processed_count}
-  Vector Search: Ready
-  BM25 Search: Ready
-  Page Images: {len(document_page_images)}

You can now ask questions about this document.
""",
            document_name
        )

    except pymupdf.FileDataError:

        return (
            " Invalid or corrupted PDF file.",
            None
        )

    except Exception as e:

        print("\nPROCESSING ERROR")
        print(traceback.format_exc())

        return (
            f"""
 **Document processing failed**

**Error:** `{str(e)}`

Try:

1. Uploading another PDF.
2. Using a smaller PDF.
3. Checking your Gemini API quota.
""",
            None
        )

In [67]:
# RESET ACTIVE DOCUMENT

def reset_document():

    global active_collection
    global active_searchable_chunks
    global active_bm25
    global active_page_images
    global active_document_name

    try:

        # Delete active Chroma collection
        if active_collection is not None:

            collection_name = active_collection.name

            try:

                chroma_client.delete_collection(
                    name=collection_name
                )

            except Exception as e:

                print(
                    "Collection cleanup warning:",
                    str(e)
                )

        # Clear active variables
        active_collection = None
        active_searchable_chunks = None
        active_bm25 = None
        active_page_images = None
        active_document_name = None

        return (
            """
#  Document Reset

The current document and its search indexes have been cleared.

Upload a new PDF to continue.
""",
            None,
            "",
            "",
            ""
        )

    except Exception as e:

        return (
            f" Reset completed with warning: {str(e)}",
            None,
            "",
            "",
            ""
        )

In [68]:
# IMPROVED QUESTION HANDLER

def ask_documind_ui(question):

    global active_collection
    global active_document_name

    try:

        if active_collection is None:

            return (
                " **No document is currently active.**\n\n"
                "Please upload and process a PDF first.",
                ""
            )

        if not question or not question.strip():

            return (
                " Please enter a question.",
                ""
            )

        response = ask_uploaded_document(
            question.strip()
        )

        answer = response.get(
            "answer",
            "No answer could be generated."
        )

        sources = response.get(
            "sources",
            []
        )

        source_text = "###  Sources\n\n"

        if sources:

            seen = set()

            for source in sources:

                document_name = source.get(
                    "document_name",
                    active_document_name
                )

                page_number = source.get(
                    "page_number",
                    "Unknown"
                )

                source_key = (
                    document_name,
                    page_number
                )

                if source_key not in seen:

                    source_text += (
                        f"-  **{document_name}** "
                        f"— Page {page_number}\n"
                    )

                    seen.add(source_key)

        else:

            source_text += (
                "No source pages were returned."
            )

        return (
            answer,
            source_text
        )

    except Exception as e:

        error_message = str(e)

        print("\nQUESTION ERROR")
        print(traceback.format_exc())

        if (
            "429" in error_message
            or "RESOURCE_EXHAUSTED" in error_message
        ):

            return (
                """
 **Gemini API quota reached.**

Please wait a while and try again.

The uploaded document is still available, so you do not need to process it again.
""",
                ""
            )

        return (
            f"""
 **Unable to answer the question**

Error: `{error_message}`

Please try again.
""",
            ""
        )

In [69]:
# DOCUMIND AI — IMPROVED GRADIO UI

import gradio as gr


def process_pdf_ui(pdf_file):

    status, document_name = process_uploaded_pdf(
        pdf_file
    )

    return (
        status,
        document_name or ""
    )


with gr.Blocks(
    title="DocuMind AI"
) as demo:

    gr.Markdown("""
#  DocuMind AI

### Multimodal Document Intelligence RAG

Upload any PDF and ask intelligent questions about its content.

**Features**

-  Gemini-powered RAG
-  Semantic Search
-  BM25 Keyword Search
-  Hybrid Retrieval
-  Table & Document Understanding
-  PDF Page Analysis
-  Page-level Sources
""")


    # DOCUMENT UPLOAD SECTION

    with gr.Row():

        pdf_input = gr.File(
            label=" Upload PDF",
            file_types=[".pdf"],
            type="filepath"
        )

        with gr.Column():

            process_button = gr.Button(
                " Process Document"
            )

            reset_button = gr.Button(
                " Reset Document"
            )


    active_document_display = gr.Textbox(
        label="Active Document",
        interactive=False
    )


    processing_status = gr.Markdown(
        value="Upload a PDF to begin."
    )


    # QUESTION SECTION

    gr.Markdown("##  Ask Questions")

    question_input = gr.Textbox(
        label="Your Question",
        placeholder=(
            "Example: What was the revenue growth "
            "and what factors contributed to it?"
        ),
        lines=3
    )


    ask_button = gr.Button(
        " Ask DocuMind AI"
    )


    # ANSWER SECTION

    answer_output = gr.Markdown(
        value="Your answer will appear here."
    )

    sources_output = gr.Markdown(
        value=""
    )


    # PROCESS DOCUMENT EVENT

    process_button.click(

        fn=process_pdf_ui,

        inputs=pdf_input,

        outputs=[
            processing_status,
            active_document_display
        ]
    )


    # ASK QUESTION EVENT

    ask_button.click(

        fn=ask_documind_ui,

        inputs=question_input,

        outputs=[
            answer_output,
            sources_output
        ]
    )


    # RESET DOCUMENT EVENT

    reset_button.click(

    fn=reset_document,

    inputs=[],

    outputs=[
        processing_status,
        pdf_input,
        question_input,
        answer_output,
        sources_output
    ]
).then(

    fn=lambda: "",

    inputs=[],

    outputs=active_document_display
)

In [70]:
demo.queue()

demo.launch(
    share=True,
    debug=False
)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://009d0158ebb5ac80d5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
